# Final Zindi test-set benchmark

Runs the five selected systems on `resources/Test.csv` in a fixed order while keeping only one neural model in GPU memory at a time:

1. base GLM-OCR, greedy decoding
2. fine-tuned GLM-OCR checkpoint 642, greedy decoding
3. the same fine-tuned GLM-OCR checkpoint with beam 4 + character KenLM (lambda 0.10)
4. base Qwen3-VL-4B-Instruct, greedy decoding
5. fine-tuned Qwen3-VL-4B checkpoint 257, greedy decoding

Predictions are cached after every image. As soon as one model variant finishes, its detailed `predictions_<variant>.csv` and Zindi-ready `submission_<variant>.csv` files are saved. After all variants finish, the combined `training_outputs/final_test_benchmark/benchmark.csv` is assembled with model name, cumulative training steps, cumulative training epochs, checkpoint, decoder, ID, and prediction. The test set has no public labels, so leaderboard scores must be recorded separately after upload.

The Qwen checkpoint is intentionally the latest completed checkpoint (Stage 2 step 257) even though that training stage was interrupted later. This notebook performs inference only.

In [ ]:
import subprocess, sys
from pathlib import Path

INSTALL_DIRECTORY = Path.cwd().resolve()
INSTALL_ROOT = next((path for path in (INSTALL_DIRECTORY, *INSTALL_DIRECTORY.parents)
                     if (path / 'requirements-benchmark.txt').is_file()
                     and (path / 'requirements-kenlm.txt').is_file()), None)
assert INSTALL_ROOT is not None, 'Could not locate the project requirements files.'
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install',
    '-r', str(INSTALL_ROOT / 'requirements-benchmark.txt'),
    '-r', str(INSTALL_ROOT / 'requirements-kenlm.txt'),
])


In [ ]:
from __future__ import annotations

import gc, hashlib, json, math, sys, time, unicodedata
from pathlib import Path

import kenlm
import pandas as pd
import torch
from IPython.display import display
from PIL import Image, ImageOps
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import (
    AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig,
    GlmOcrForConditionalGeneration, LogitsProcessor, LogitsProcessorList,
)

WORKING_DIRECTORY = Path.cwd().resolve()
ROOT = next((path for path in (WORKING_DIRECTORY, *WORKING_DIRECTORY.parents)
             if (path / 'resources/Test.csv').is_file()), None)
assert ROOT is not None, 'Could not locate the project root or resources/Test.csv.'
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

IMAGES = ROOT / 'resources/images'
GLM_BASE = ROOT / 'model/glm-ocr'
GLM_RUN = ROOT / 'training_outputs/glm_ocr_from_base_curriculum_v7_native_ocr_balanced'
GLM_STAGE1 = GLM_RUN / 'stage1/checkpoints/checkpoint-2879'
GLM_ADAPTER = GLM_RUN / 'stage2/checkpoints/checkpoint-642'
KENLM_PATH = GLM_RUN / 'current_checkpoint_kenlm/train_only_char6/train_labels_char6.binary'
QWEN_BASE = ROOT / 'model/qwen3-vl-4b-instruct'
QWEN_RUN = ROOT / 'training_outputs/qwen3_vl_4b_curriculum_seed1705'
QWEN_STAGE1 = QWEN_RUN / 'stage1/checkpoints/checkpoint-864'
QWEN_ADAPTER = QWEN_RUN / 'stage2/checkpoints/checkpoint-257'
OUT = ROOT / 'training_outputs/final_test_benchmark'
OUT.mkdir(parents=True, exist_ok=True)

for required in (GLM_BASE / 'config.json', GLM_ADAPTER / 'adapter_model.safetensors',
                 KENLM_PATH, QWEN_BASE / 'config.json', QWEN_ADAPTER / 'adapter_model.safetensors'):
    assert required.is_file(), f'Missing required model artifact: {required}'
assert torch.cuda.is_available(), 'A CUDA GPU is required for this benchmark.'

test = pd.read_csv(ROOT / 'resources/Test.csv', dtype={'ID': str}, keep_default_na=False)
template = pd.read_csv(ROOT / 'resources/SampleSubmission.csv', dtype={'ID': str}, keep_default_na=False)
assert test.columns.tolist() == ['ID'] and test.ID.is_unique and len(test) > 0
assert template.columns.tolist() == ['ID', 'Target'] and template.ID.tolist() == test.ID.tolist()
assert all((IMAGES / f'{image_id}.jpg').is_file() for image_id in test.ID), 'A test image is missing.'

def trainer_progress(path):
    state = json.loads((Path(path) / 'trainer_state.json').read_text(encoding='utf-8'))
    return int(state['global_step']), float(state['epoch'])

glm_stage1_steps, glm_stage1_epochs = trainer_progress(GLM_STAGE1)
glm_stage2_steps, glm_stage2_epochs = trainer_progress(GLM_ADAPTER)
qwen_stage1_steps, qwen_stage1_epochs = trainer_progress(QWEN_STAGE1)
qwen_stage2_steps, qwen_stage2_epochs = trainer_progress(QWEN_ADAPTER)

MODEL_SPECS = {
    'base_glm': {
        'model_name': 'Base GLM-OCR', 'family': 'glm', 'adapter': None, 'decoder': 'greedy',
        'training_steps': 0, 'training_epochs': 0.0, 'checkpoint': 'zai-org/GLM-OCR',
    },
    'finetuned_glm': {
        'model_name': 'Fine-tuned GLM-OCR', 'family': 'glm', 'adapter': GLM_ADAPTER, 'decoder': 'greedy',
        'training_steps': glm_stage1_steps + glm_stage2_steps,
        'training_epochs': glm_stage1_epochs + glm_stage2_epochs,
        'checkpoint': GLM_ADAPTER.relative_to(ROOT).as_posix(),
    },
    'finetuned_glm_kenlm': {
        'model_name': 'Fine-tuned GLM-OCR + KenLM', 'family': 'glm', 'adapter': GLM_ADAPTER,
        'decoder': 'beam4 + char-6gram KenLM lambda=0.10',
        'training_steps': glm_stage1_steps + glm_stage2_steps,
        'training_epochs': glm_stage1_epochs + glm_stage2_epochs,
        'checkpoint': GLM_ADAPTER.relative_to(ROOT).as_posix(),
    },
    'base_qwen': {
        'model_name': 'Base Qwen3-VL-4B-Instruct', 'family': 'qwen', 'adapter': None, 'decoder': 'greedy',
        'training_steps': 0, 'training_epochs': 0.0, 'checkpoint': 'Qwen/Qwen3-VL-4B-Instruct',
    },
    'finetuned_qwen': {
        'model_name': 'Fine-tuned Qwen3-VL-4B', 'family': 'qwen', 'adapter': QWEN_ADAPTER, 'decoder': 'greedy',
        'training_steps': qwen_stage1_steps + qwen_stage2_steps,
        'training_epochs': qwen_stage1_epochs + qwen_stage2_epochs,
        'checkpoint': QWEN_ADAPTER.relative_to(ROOT).as_posix(),
    },
}
MODELS_TO_RUN = tuple(MODEL_SPECS)  # A subset can be used for resumable partial runs.
MAX_NEW_TOKENS = 128
GLM_MIN_PIXELS, GLM_MAX_PIXELS = 384 * 28 * 28, 1536 * 28 * 28
QWEN_MIN_PIXELS, QWEN_MAX_PIXELS = 65536, 393216
GLM_PROMPT = 'Text Recognition:'
QWEN_PROMPT = ('Transcribe the handwritten text exactly. Preserve original spelling, capitalization, '
               'punctuation, abbreviations, and symbols. Output only the transcription.')
KENLM_LAMBDA, KENLM_TOP_K = 0.10, 256
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
DEVICE = torch.device('cuda:0')
display(pd.DataFrame([{k: v for k, v in spec.items() if k != 'adapter'}
                      for spec in MODEL_SPECS.values()]))


## Shared preprocessing, model loading, and decoding

Base and fine-tuned variants within each architecture use identical preprocessing and prompts. Qwen loads in the same NF4 configuration used during QLoRA validation.

In [ ]:
def normalize(value):
    return unicodedata.normalize('NFC', str(value)).strip()

def file_hash(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def object_hash(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()

def write_json(path, value):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')
    temporary.replace(path)

def write_csv(path, frame):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temporary, index=False)
    temporary.replace(path)

def load_image(image_id, min_pixels, max_pixels):
    with Image.open(IMAGES / f'{image_id}.jpg') as source:
        image = ImageOps.exif_transpose(source).convert('RGB')
    for bound, comparison in ((max_pixels, lambda area, limit: area > limit),
                              (min_pixels, lambda area, limit: area < limit)):
        if comparison(image.width * image.height, bound):
            scale = math.sqrt(bound / (image.width * image.height))
            image = image.resize((max(1, int(image.width * scale)),
                                  max(1, int(image.height * scale))), Image.Resampling.BICUBIC)
    return image

def load_glm_runtime(spec):
    processor = AutoProcessor.from_pretrained(GLM_BASE, local_files_only=True, use_fast=False)
    base = GlmOcrForConditionalGeneration.from_pretrained(
        GLM_BASE, local_files_only=True, dtype=DTYPE, device_map={'': 0}, attn_implementation='sdpa')
    model = (PeftModel.from_pretrained(base, spec['adapter'], is_trainable=False)
             if spec['adapter'] is not None else base).eval()
    generation = json.loads((GLM_BASE / 'generation_config.json').read_text(encoding='utf-8'))
    eos_ids = generation['eos_token_id']
    eos_ids = [eos_ids] if isinstance(eos_ids, int) else list(eos_ids)
    eos_ids = sorted(set(eos_ids + [processor.tokenizer.eos_token_id]))
    character_lm = kenlm.Model(str(KENLM_PATH)) if spec['decoder'].startswith('beam4') else None
    return {'model': model, 'processor': processor, 'eos_ids': eos_ids, 'character_lm': character_lm}

def load_qwen_runtime(spec):
    processor = AutoProcessor.from_pretrained(QWEN_BASE, local_files_only=True, trust_remote_code=True)
    quantization = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=DTYPE)
    base = AutoModelForImageTextToText.from_pretrained(
        QWEN_BASE, local_files_only=True, trust_remote_code=True, quantization_config=quantization,
        device_map={'': 0}, dtype=DTYPE, attn_implementation='sdpa')
    model = (PeftModel.from_pretrained(base, spec['adapter'], is_trainable=False)
             if spec['adapter'] is not None else base).eval()
    model.config.use_cache = True
    return {'model': model, 'processor': processor}

class CharacterKenLMFusion(LogitsProcessor):
    def __init__(self, processor, character_lm, eos_ids, prompt_length):
        self.processor = processor; self.character_lm = character_lm
        self.eos_ids = set(map(int, eos_ids)); self.prompt_length = int(prompt_length)
        self.special_ids = set(processor.tokenizer.all_special_ids)
        self.pieces, self.states, self.transitions = {}, {}, {}
        state = kenlm.State(); character_lm.BeginSentenceWrite(state); self.states[()] = state

    def piece(self, token_id):
        token_id = int(token_id)
        if token_id not in self.pieces:
            piece = None if token_id in self.special_ids else unicodedata.normalize(
                'NFC', self.processor.tokenizer.decode(
                    [token_id], skip_special_tokens=False, clean_up_tokenization_spaces=False))
            self.pieces[token_id] = None if not piece or '\ufffd' in piece else piece
        return self.pieces[token_id]

    def advance(self, state, text):
        total, current = 0.0, state
        for character in text:
            output = kenlm.State()
            total += self.character_lm.BaseScore(current, f'U{ord(character):06X}', output)
            current = output
        return total, current

    def state_for(self, prefix):
        if prefix not in self.states:
            piece = self.piece(prefix[-1])
            if piece is None:
                raise RuntimeError(f'Unsupported active token: {prefix[-1]}')
            _, self.states[prefix] = self.advance(self.state_for(prefix[:-1]), piece)
        return self.states[prefix]

    def delta(self, prefix, token_id):
        key = (prefix, int(token_id))
        if key in self.transitions:
            return self.transitions[key]
        state = self.state_for(prefix)
        if token_id in self.eos_ids:
            output = kenlm.State(); value = self.character_lm.BaseScore(state, '</s>', output)
        else:
            piece = self.piece(token_id)
            if piece is None:
                return None
            value, _ = self.advance(state, piece)
        self.transitions[key] = float(value)
        return float(value)

    def __call__(self, input_ids, scores):
        candidates_by_row = torch.topk(scores, k=min(KENLM_TOP_K, scores.shape[-1]), dim=-1).indices.tolist()
        fused = torch.full_like(scores, -torch.inf)
        for row_index, token_ids in enumerate(candidates_by_row):
            prefix = tuple(map(int, input_ids[row_index, self.prompt_length:].detach().cpu().tolist()))
            valid, bonuses = [], []
            for token_id in set(token_ids) | self.eos_ids:
                bonus = self.delta(prefix, token_id)
                if bonus is not None:
                    valid.append(token_id); bonuses.append(bonus)
            index = torch.tensor(valid, device=scores.device, dtype=torch.long)
            fused[row_index, index] = scores[row_index, index] + KENLM_LAMBDA * math.log(10.0) * torch.tensor(
                bonuses, device=scores.device, dtype=scores.dtype)
        return fused

def predict_glm(runtime, image_id, use_kenlm=False):
    processor, model = runtime['processor'], runtime['model']
    image = load_image(image_id, GLM_MIN_PIXELS, GLM_MAX_PIXELS)
    prompt = ('[gMASK]<sop><|user|>\n<|begin_of_image|><|image|><|end_of_image|>'
              + GLM_PROMPT + '<|assistant|>')
    inputs = processor(text=[prompt], images=[image], add_special_tokens=False, return_tensors='pt')
    inputs.pop('token_type_ids', None); prompt_length = int(inputs['input_ids'].shape[1]); inputs = inputs.to(DEVICE)
    processors = LogitsProcessorList()
    if use_kenlm:
        processors.append(CharacterKenLMFusion(
            processor, runtime['character_lm'], runtime['eos_ids'], prompt_length))
    options = {'max_new_tokens': MAX_NEW_TOKENS, 'num_beams': 4 if use_kenlm else 1,
               'do_sample': False, 'use_cache': True, 'eos_token_id': runtime['eos_ids'],
               'pad_token_id': processor.tokenizer.pad_token_id}
    if use_kenlm:
        options.update(length_penalty=1.0, early_stopping=True, logits_processor=processors,
                       renormalize_logits=False)
    with torch.inference_mode():
        output = model.generate(**inputs, **options)
    generated = output[0, prompt_length:].tolist()
    eos_index = next((i for i, token in enumerate(generated) if token in runtime['eos_ids']), None)
    if eos_index is not None:
        generated = generated[:eos_index + 1]
    prediction = normalize(processor.decode(
        generated, skip_special_tokens=True, clean_up_tokenization_spaces=False))
    return {'Target': prediction, 'generated_tokens': len(generated),
            'hit_token_limit': eos_index is None and len(generated) >= MAX_NEW_TOKENS}

def predict_qwen(runtime, image_id):
    processor, model = runtime['processor'], runtime['model']
    image = load_image(image_id, QWEN_MIN_PIXELS, QWEN_MAX_PIXELS)
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image}, {'type': 'text', 'text': QWEN_PROMPT}]}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt')
    inputs.pop('token_type_ids', None); prompt_length = int(inputs['input_ids'].shape[1]); inputs = inputs.to(DEVICE)
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                                num_beams=1, do_sample=False, use_cache=True)
    generated = output[0, prompt_length:].tolist()
    prediction = normalize(processor.decode(
        generated, skip_special_tokens=True, clean_up_tokenization_spaces=False))
    eos_ids = model.generation_config.eos_token_id
    eos_ids = {eos_ids} if isinstance(eos_ids, int) else set(eos_ids or [])
    return {'Target': prediction, 'generated_tokens': len(generated),
            'hit_token_limit': len(generated) >= MAX_NEW_TOKENS and
                               (not generated or generated[-1] not in eos_ids)}


## Run sequential inference

This is the only long-running cell. Rerunning resumes from compatible per-image caches. To run fewer models in one session, change `MODELS_TO_RUN` above; run the remaining models later before assembling the final CSV.

In [ ]:
glm_manifest = json.loads((GLM_RUN / 'run_identity.json').read_text(encoding='utf-8'))
qwen_manifest = json.loads((QWEN_RUN / 'run_identity.json').read_text(encoding='utf-8'))
signatures = {}
for model_key, spec in MODEL_SPECS.items():
    identity = {
        'model_key': model_key, 'checkpoint': spec['checkpoint'], 'decoder': spec['decoder'],
        'glm_run': glm_manifest['identity_key'] if spec['family'] == 'glm' else None,
        'qwen_revision': qwen_manifest['identity']['model_revision'] if spec['family'] == 'qwen' else None,
        'adapter_sha256': file_hash(spec['adapter'] / 'adapter_model.safetensors')
                          if spec['adapter'] is not None else None,
        'kenlm_sha256': file_hash(KENLM_PATH) if model_key == 'finetuned_glm_kenlm' else None,
        'max_new_tokens': MAX_NEW_TOKENS,
    }
    signatures[model_key] = object_hash(identity)

def cache_path(model_key, image_id):
    return OUT / 'cache' / model_key / f'{image_id}.json'

def cached_prediction(model_key, image_id):
    path = cache_path(model_key, image_id)
    if not path.is_file():
        return None
    row = json.loads(path.read_text(encoding='utf-8'))
    expected_image_hash = file_hash(IMAGES / f'{image_id}.jpg')
    return row if row.get('signature') == signatures[model_key] and row.get('image_sha256') == expected_image_hash else None

def save_variant_outputs(model_key):
    predictions = [cached_prediction(model_key, image_id) for image_id in test.ID]
    assert all(row is not None for row in predictions), f'{model_key} is not complete.'
    prediction_frame = pd.DataFrame([{
        'ID': row['ID'], 'Target': row['Target'],
        'generated_tokens': row['generated_tokens'],
        'hit_token_limit': row['hit_token_limit'],
        'inference_seconds': row['seconds'],
    } for row in predictions])
    prediction_path = OUT / f'predictions_{model_key}.csv'
    write_csv(prediction_path, prediction_frame)
    submission = prediction_frame[['ID', 'Target']].copy()
    submission_path = OUT / f'submission_{model_key}.csv'
    write_csv(submission_path, submission)
    write_json(OUT / f'{model_key}_predictions.json', {
        'model_key': model_key, 'signature': signatures[model_key],
        'rows': len(prediction_frame),
        'predictions_file': prediction_path.name,
        'predictions_sha256': file_hash(prediction_path),
        'submission_file': submission_path.name,
        'submission_sha256': file_hash(submission_path),
    })
    return prediction_path, submission_path

assert set(MODELS_TO_RUN).issubset(MODEL_SPECS)
for model_key in MODELS_TO_RUN:
    spec = MODEL_SPECS[model_key]
    pending = [image_id for image_id in test.ID if cached_prediction(model_key, image_id) is None]
    print(f'{model_key}: {len(test) - len(pending)}/{len(test)} cached')
    if pending:
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
        runtime = load_glm_runtime(spec) if spec['family'] == 'glm' else load_qwen_runtime(spec)
        started = time.perf_counter()
        try:
            for image_id in tqdm(pending, desc=model_key):
                torch.cuda.synchronize(); item_started = time.perf_counter()
                result = (predict_glm(runtime, image_id, use_kenlm=model_key == 'finetuned_glm_kenlm')
                          if spec['family'] == 'glm' else predict_qwen(runtime, image_id))
                torch.cuda.synchronize()
                if not result['Target'].strip():
                    raise RuntimeError(f'Blank prediction from {model_key} for {image_id}; inspect before continuing.')
                write_json(cache_path(model_key, image_id), {
                    'signature': signatures[model_key], 'image_sha256': file_hash(IMAGES / f'{image_id}.jpg'),
                    'ID': image_id, 'seconds': time.perf_counter() - item_started, **result})
        finally:
            del runtime
            gc.collect(); torch.cuda.empty_cache()
        write_json(OUT / f'{model_key}_runtime.json', {
            'model_key': model_key, 'new_images': len(pending),
            'wall_seconds': time.perf_counter() - started,
            'peak_cuda_gib': torch.cuda.max_memory_allocated() / 1024**3})
    prediction_path, submission_path = save_variant_outputs(model_key)
    print(f'Saved {model_key}: {prediction_path.name}, {submission_path.name}')


## Assemble `benchmark.csv` and Zindi submissions


In [ ]:
benchmark_rows, model_rows = [], []
for model_key, spec in MODEL_SPECS.items():
    predictions = [cached_prediction(model_key, image_id) for image_id in test.ID]
    assert all(row is not None for row in predictions), f'Run or resume {model_key} before assembling results.'
    submission = template[['ID']].copy()
    submission['Target'] = [row['Target'] for row in predictions]
    assert submission.Target.notna().all() and submission.Target.str.strip().ne('').all()
    submission_path = OUT / f'submission_{model_key}.csv'
    submission.to_csv(submission_path, index=False)
    model_row = {
        'model_key': model_key, 'model_name': spec['model_name'],
        'training_steps': spec['training_steps'], 'training_epochs': spec['training_epochs'],
        'checkpoint': spec['checkpoint'], 'decoding': spec['decoder'],
        'submission_file': submission_path.name,
    }
    model_rows.append(model_row)
    for prediction in predictions:
        benchmark_rows.append({
            **model_row, 'ID': prediction['ID'], 'Target': prediction['Target'],
            'generated_tokens': prediction['generated_tokens'],
            'hit_token_limit': prediction['hit_token_limit'], 'inference_seconds': prediction['seconds'],
        })

benchmark = pd.DataFrame(benchmark_rows)
assert len(benchmark) == len(test) * len(MODEL_SPECS)
assert not benchmark.duplicated(['model_key', 'ID']).any()
benchmark_path = OUT / 'benchmark.csv'
models_path = OUT / 'benchmark_models.csv'
benchmark.to_csv(benchmark_path, index=False)
pd.DataFrame(model_rows).to_csv(models_path, index=False)
write_json(OUT / 'benchmark_manifest.json', {
    'rows': len(benchmark), 'test_images': len(test), 'models': model_rows,
    'benchmark_sha256': file_hash(benchmark_path),
    'note': 'Zindi leaderboard scores are external and are not fabricated by this notebook.',
})
display(pd.DataFrame(model_rows))
display(benchmark.head())
print('Saved:', benchmark_path)
print('Zindi submissions:', *(OUT / row['submission_file'] for row in model_rows), sep='\n- ')


## Official Zindi results

These scores were returned by Zindi after uploading each independently generated submission. Higher is better. They are stored separately from the prediction benchmark because the private test labels are unavailable locally.

In [ ]:
ZINDI_SCORES = {
    'base_glm': 0.716085382,
    'finetuned_glm': 0.885003890,
    'finetuned_glm_kenlm': 0.895312889,
    'base_qwen': 0.572675155,
    'finetuned_qwen': 0.845694646,
}
assert set(ZINDI_SCORES) == set(MODEL_SPECS)
results = pd.DataFrame(model_rows)
results['zindi_score'] = results['model_key'].map(ZINDI_SCORES)
base_scores = {'glm': ZINDI_SCORES['base_glm'], 'qwen': ZINDI_SCORES['base_qwen']}
results['gain_vs_family_base'] = [
    score - base_scores[MODEL_SPECS[key]['family']]
    for key, score in zip(results['model_key'], results['zindi_score'])
]
results_path = OUT / 'benchmark_results.csv'
write_csv(results_path, results)
display(results[['model_name', 'training_steps', 'decoding', 'zindi_score', 'gain_vs_family_base']])
print('Saved:', results_path)


## Official-score comparison

The figure is saved under `figures/` because it contains only aggregate results and is safe to publish.

In [ ]:
import matplotlib.pyplot as plt

PUBLIC_FIGURES = ROOT / 'figures'
PUBLIC_FIGURES.mkdir(parents=True, exist_ok=True)
plot_results = results.sort_values('zindi_score', ascending=True)
colors = [('#315a8a' if MODEL_SPECS[key]['family'] == 'glm' else '#c96b32')
          for key in plot_results['model_key']]
fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.barh(plot_results['model_name'], plot_results['zindi_score'], color=colors)
ax.bar_label(bars, labels=[f'{score:.4f}' for score in plot_results['zindi_score']], padding=5)
ax.set_xlim(0, 1)
ax.set_xlabel('Official Zindi score (higher is better)')
ax.set_title('Base and fine-tuned OCR models on the Zindi test set')
ax.grid(axis='x', alpha=0.2)
ax.spines[['top', 'right', 'left']].set_visible(False)
fig.tight_layout()
score_plot_path = PUBLIC_FIGURES / 'zindi_score_comparison.png'
fig.savefig(score_plot_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', score_plot_path)


## GLM versus Qwen training steps

This compares cumulative optimizer steps for the selected fine-tuned checkpoints. Steps are useful run metadata, but they are not a hardware- or compute-normalized measure because the architectures and training recipes differ.

In [ ]:
step_keys = ['finetuned_glm', 'finetuned_qwen']
step_labels = [MODEL_SPECS[key]['model_name'] for key in step_keys]
step_values = [MODEL_SPECS[key]['training_steps'] for key in step_keys]
fig, ax = plt.subplots(figsize=(7.5, 4.5))
bars = ax.bar(step_labels, step_values, color=['#315a8a', '#c96b32'], width=0.6)
ax.bar_label(bars, labels=[f'{steps:,}' for steps in step_values], padding=5)
ax.set_ylabel('Cumulative optimizer steps')
ax.set_title('Selected fine-tuned checkpoints')
ax.grid(axis='y', alpha=0.2)
ax.spines[['top', 'right']].set_visible(False)
fig.tight_layout()
steps_plot_path = PUBLIC_FIGURES / 'training_steps_comparison.png'
fig.savefig(steps_plot_path, dpi=200, bbox_inches='tight')
plt.show()
print('Saved:', steps_plot_path)


## Inspect one image and its transcription

Choose any completed model and test ID below. The preview is written only under ignored `training_outputs/`; do not publish it because the competition images and predictions are governed by the Zindi data terms. This cell reuses the saved prediction and does not reload the model.

In [ ]:
import textwrap
import matplotlib.pyplot as plt
DEMO_MODEL_KEY = 'finetuned_glm_kenlm'
DEMO_IMAGE_ID = str(test.ID.iloc[50])
assert DEMO_MODEL_KEY in MODEL_SPECS
assert DEMO_IMAGE_ID in set(test.ID)
demo = cached_prediction(DEMO_MODEL_KEY, DEMO_IMAGE_ID)
assert demo is not None, f'Run {DEMO_MODEL_KEY} for {DEMO_IMAGE_ID} first.'
with Image.open(IMAGES / f'{DEMO_IMAGE_ID}.jpg') as source:
    demo_image = ImageOps.exif_transpose(source).convert('RGB')
fig_width = 14
fig_height = max(3.0, fig_width * demo_image.height / max(demo_image.width, 1) + 1.8)
fig, ax = plt.subplots(figsize=(fig_width, fig_height))
ax.imshow(demo_image)
ax.axis('off')
wrapped_prediction = textwrap.fill(demo['Target'], width=110)
ax.set_title(
    f"{MODEL_SPECS[DEMO_MODEL_KEY]['model_name']} — \n{wrapped_prediction}",
    fontsize=12, pad=14, loc='left',
)
fig.tight_layout()
private_figure_dir = OUT / 'figures'
private_figure_dir.mkdir(parents=True, exist_ok=True)
preview_path = private_figure_dir / f'{DEMO_MODEL_KEY}_{DEMO_IMAGE_ID}.png'
fig.savefig(preview_path, dpi=160, bbox_inches='tight')
plt.show()
print('Prediction:', demo['Target'])
print('Saved private preview:', preview_path)
